CP2 - Dynamic Programming

Integrantes do Grupo:

Eduardo Santiago Bassan - RM561474
Henry Browne Andrade - RM562622
Mariana S. do Egito Moreira - RM562544
João Victor de Souza Abe - RM561446
Deivid Ruan - RM566356

# IMPORTANDO AS BIBLIOTECAS

In [11]:
!pip install osmnx folium networkx requests matplotlib pandas --q

In [57]:
import os
import requests
import pandas as pd
import networkx as nx
import osmnx as ox
import folium
import matplotlib.pyplot as plt
import math
from functools import lru_cache
import time
import tracemalloc

plt.rcParams["figure.figsize"] = (10,10)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 11

ox.settings.use_cache = True
ox.settings.log_console = False

print("Bibliotecas carregadas com sucesso.")


Bibliotecas carregadas com sucesso.


In [14]:
!pip install scikit-learn

# SÃO PAULO - BRASIL

In [24]:
# Criando nós (estações)

estacoes = {
    "Tucuruvi" : (-23.4800, -46.6033),
    "Parada Inglesa" : (-23.4870, -46.6086),
    "Carandiru" : (-23.5092, -46.6246),
    "Luz" : (-23.5365, -46.6333),
    "Se" : (-23.5503, -46.6342),
    "Liberdade" : (-23.5555, -46.6356),
    "São Joaquim" : (-23.5614, -46.6380),
    "Paraiso" : (-23.5746, -46.6407),
    "Santa Cruz" : (-23.5985, -46.6388),
    "Moema" : (-23.6033, -46.6610),
    "Campo Belo" : (-23.6260, -46.6564),
    "Santo Amaro" : (-23.6557, -46.7100),
    "Capao Redondo" : (-23.6603, -46.7702),
    "Anhengabau" : (-23.5475, -46.6377),
    "Consolacao": (-23.5583, -46.6605),
    "Faria Lima" : (-23.5672, -46.6933),
    "Pinheiros" : (-23.5679, -46.7017),
    "Cidade Jardin" : (-23.5732, -46.6997),
    "Morumbi" : (-23.6008, -46.7220),
    "Joao Dias" : (-23.6433, -46.7463),
}

df_estacoes = pd.DataFrame(
    [
        {"local" : nome , "lat" : coord[0], "lon" : coord[1]}
        for nome, coord in estacoes.items()
    ]
)

print(df_estacoes)

             local      lat      lon
0         Tucuruvi -23.4800 -46.6033
1   Parada Inglesa -23.4870 -46.6086
2        Carandiru -23.5092 -46.6246
3              Luz -23.5365 -46.6333
4               Se -23.5503 -46.6342
5        Liberdade -23.5555 -46.6356
6      São Joaquim -23.5614 -46.6380
7          Paraiso -23.5746 -46.6407
8       Santa Cruz -23.5985 -46.6388
9            Moema -23.6033 -46.6610
10      Campo Belo -23.6260 -46.6564
11     Santo Amaro -23.6557 -46.7100
12   Capao Redondo -23.6603 -46.7702
13      Anhengabau -23.5475 -46.6377
14      Consolacao -23.5583 -46.6605
15      Faria Lima -23.5672 -46.6933
16       Pinheiros -23.5679 -46.7017
17   Cidade Jardin -23.5732 -46.6997
18         Morumbi -23.6008 -46.7220
19       Joao Dias -23.6433 -46.7463


In [25]:
# Grafo apenas com estações

G_estacoes = nx.Graph()
for nome, (lat, lon) in estacoes.items():
    G_estacoes.add_node(nome, pos=(lat, lon))

print(G_estacoes)

print(G_estacoes.nodes)

Graph with 20 nodes and 0 edges
['Tucuruvi', 'Parada Inglesa', 'Carandiru', 'Luz', 'Se', 'Liberdade', 'São Joaquim', 'Paraiso', 'Santa Cruz', 'Moema', 'Campo Belo', 'Santo Amaro', 'Capao Redondo', 'Anhengabau', 'Consolacao', 'Faria Lima', 'Pinheiros', 'Cidade Jardin', 'Morumbi', 'Joao Dias']


Grafo não direcionado, porque:
- o metrô permite ida e volta entre estações
- o custo é simétrico (mesma distância)

In [26]:
# Criando as arestas

def distancia(a, b):
    return math.sqrt((a[0]-b[0])**2 + (a[1]-b[1])**2)

def conectar(a, b):
    dist = distancia(estacoes[a], estacoes[b])
    G_estacoes.add_edge(a, b, weight=dist)

# Linha 1
conectar("Tucuruvi", "Parada Inglesa")
conectar("Parada Inglesa", "Carandiru")
conectar("Carandiru", "Luz")
conectar("Luz", "Se")
conectar("Se", "Liberdade")
conectar("Liberdade", "São Joaquim")
conectar("São Joaquim", "Paraiso")
conectar("Paraiso", "Santa Cruz")
conectar("Santa Cruz", "Moema")
conectar("Moema", "Campo Belo")
conectar("Campo Belo", "Santo Amaro")
conectar("Santo Amaro", "Capao Redondo")

# Linha 2
conectar("Tucuruvi", "Parada Inglesa")
conectar("Parada Inglesa", "Carandiru")
conectar("Carandiru", "Luz")
conectar("Luz", "Anhengabau")
conectar("Anhengabau", "Consolacao")
conectar("Consolacao", "Faria Lima")
conectar("Faria Lima", "Pinheiros")
conectar("Pinheiros", "Cidade Jardin")
conectar("Cidade Jardin", "Morumbi")
conectar("Morumbi", "Joao Dias")
conectar("Joao Dias", "Capao Redondo")

print(G_estacoes.edges)

[('Tucuruvi', 'Parada Inglesa'), ('Parada Inglesa', 'Carandiru'), ('Carandiru', 'Luz'), ('Luz', 'Se'), ('Luz', 'Anhengabau'), ('Se', 'Liberdade'), ('Liberdade', 'São Joaquim'), ('São Joaquim', 'Paraiso'), ('Paraiso', 'Santa Cruz'), ('Santa Cruz', 'Moema'), ('Moema', 'Campo Belo'), ('Campo Belo', 'Santo Amaro'), ('Santo Amaro', 'Capao Redondo'), ('Capao Redondo', 'Joao Dias'), ('Anhengabau', 'Consolacao'), ('Consolacao', 'Faria Lima'), ('Faria Lima', 'Pinheiros'), ('Pinheiros', 'Cidade Jardin'), ('Cidade Jardin', 'Morumbi'), ('Morumbi', 'Joao Dias')]


In [44]:
# Penalidade Horario

def fator_horario(hora):
    if 6 <= hora <= 9:
        return 1.5  
    elif 17 <= hora <= 20:
        return 1.7 
    else:
        return 1.0

In [50]:
# Definindo a hora 

hora_input = int(input("Insira a hora: "))

In [62]:
# Calculando o menor caminho de Tucuruvi para Capao Redondo com MEMOIZAÇÃO

@lru_cache(None)
def menor_caminho_rec(atual, destino, visitados, hora):
    if atual == destino:
        return (0, [atual])

    visitados = set(visitados)
    visitados.add(atual)

    melhor_dist = float("inf")
    melhor_caminho = []

    for vizinho in G_estacoes.neighbors(atual):
        if vizinho not in visitados:
            peso_base = G_estacoes[atual][vizinho]["weight"]
            peso = peso_base * fator_horario(hora)
            dist, caminho = menor_caminho_rec(vizinho, destino, tuple(sorted(visitados)), hora)

            if dist + peso < melhor_dist:
                melhor_dist = dist + peso
                melhor_caminho = [atual] + caminho

    return (melhor_dist, melhor_caminho)

dist, caminho_mais_curto = menor_caminho_rec("Tucuruvi", "Capao Redondo", tuple(), hora_input)
# print(dist, caminho, len(caminho))
print(f"Caminho mais curto : {caminho_mais_curto} - Total Estações : {len(caminho_mais_curto)}")

Caminho mais curto : ['Tucuruvi', 'Parada Inglesa', 'Carandiru', 'Luz', 'Anhengabau', 'Consolacao', 'Faria Lima', 'Pinheiros', 'Cidade Jardin', 'Morumbi', 'Joao Dias', 'Capao Redondo'] - Total Estações : 12


In [61]:
# Calculando o menor caminho de Tucuruvi para Capao Redondo SEM MEMOIZAÇÃO

def menor_caminho_rec_sem_memo(atual, destino, visitados, hora):
    if atual == destino:
        return (0, [atual])

    visitados = set(visitados)
    visitados.add(atual)

    melhor_dist = float("inf")
    melhor_caminho = []

    for vizinho in G_estacoes.neighbors(atual):
        if vizinho not in visitados:
            peso_base = G_estacoes[atual][vizinho]["weight"]
            peso = peso_base * fator_horario(hora)
            dist, caminho = menor_caminho_rec_sem_memo(vizinho, destino, tuple(sorted(visitados)), hora)

            if dist + peso < melhor_dist:
                melhor_dist = dist + peso
                melhor_caminho = [atual] + caminho

    return (melhor_dist, melhor_caminho)

dist, caminho_mais_curto = menor_caminho_rec_sem_memo("Tucuruvi", "Capao Redondo", tuple(), hora_input)
# print(dist, caminho, len(caminho))
print(f"Caminho mais curto : {caminho_mais_curto} - Total Estações : {len(caminho_mais_curto)}")

Caminho mais curto : ['Tucuruvi', 'Parada Inglesa', 'Carandiru', 'Luz', 'Anhengabau', 'Consolacao', 'Faria Lima', 'Pinheiros', 'Cidade Jardin', 'Morumbi', 'Joao Dias', 'Capao Redondo'] - Total Estações : 12


In [58]:
# Analisando o desempenho

def medir(func, *args):
    tracemalloc.start()
    
    inicio = time.time()
    resultado = func(*args)
    fim = time.time()
    
    memoria = tracemalloc.get_traced_memory()[1]
    tracemalloc.stop()

    return fim - inicio, memoria, resultado

In [77]:
menor_caminho_rec.cache_clear()
tempo1, mem1, _ = medir(menor_caminho_rec, "Tucuruvi", "Capao Redondo", tuple(), hora_input)
tempo2, mem2, _ = medir(menor_caminho_rec_sem_memo, "Tucuruvi", "Capao Redondo", tuple(), hora_input)

print(f"Com memo: {tempo1:.6f}s | Mem: {mem1}")
print(f"Sem memo: {tempo2:.6f}s | Mem: {mem2}")

Com memo: 0.000232s | Mem: 7744
Sem memo: 0.000190s | Mem: 7720


Para o grafo utilizado, a versão sem memoização apresentou desempenho ligeiramente superior. Isso ocorre porque o problema possui baixa complexidade e pouca repetição de subproblemas, tornando o custo adicional da memoização (armazenamento e consulta em cache) maior do que o benefício obtido.
Em grafos maiores ou mais complexos, a memoização tende a apresentar vantagens significativas.

In [56]:
if not caminho_mais_curto:
    print("Rota não disponível — mapa interativo não gerado.")
else:
    coordenadas_rota = [G_estacoes.nodes[node]["pos"] for node in caminho_mais_curto]
    coordenadas_rota_maior = [G_estacoes.nodes[node]["pos"] for node in rota_mais_longa]

    mapa = folium.Map(location=estacoes["Se"], zoom_start=12)

    # Plotar estações
    for nome, (lat, lon) in estacoes.items():
        folium.CircleMarker(
            location=[lat, lon],
            radius=8,
            fill=True,
            fill_opacity=0.9,
            popup=nome,
            tooltip=nome
        ).add_to(mapa)

    # Plotar rota mais rapida
    folium.PolyLine(
        locations=coordenadas_rota,
        color="blue",
        weight=5,
        opacity=0.8,
        tooltip="Rota calculada"
    ).add_to(mapa)

    # Plotar rota mais lenta
    folium.PolyLine(
        locations=coordenadas_rota_maior,
        color="red",
        weight=5,
        opacity=0.8,
        tooltip="Rota calculada"
    ).add_to(mapa)

if caminho_mais_curto is not None:
    nome_arquivo_html = "mapa_grafo_estacoes.html"
    mapa.save(nome_arquivo_html)
    print(f"Mapa salvo em: {nome_arquivo_html}")
else:
    print("Mapa não disponível para salvar.")

Mapa salvo em: mapa_grafo_estacoes.html
